# Imports

In [2]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import TrainingArguments
import evaluate
from transformers import Trainer
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Current device:", torch.cuda.current_device())
else:
    print("Running on CPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4080
Current device: 0


In [4]:
import ast

data = []

with open("../ASTE dataset/14lap/train.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split("####")

        # we only care about last part (annotations)
        if len(parts) < 3:
            continue  # skip malformed lines

        sentence = parts[0]   # or keep raw text
        annotations_str = parts[-1]

        try:
            annotations = ast.literal_eval(annotations_str)
        except:
            print("Skipping bad annotation:", annotations_str)
            continue

        data.append({
            "sentence": sentence,
            "annotations": annotations
        })

df = pd.DataFrame(data)
print(df.head())

                                            sentence  \
0  I charge it at night and skip taking the cord ...   
1  it is of high quality , has a killer GUI , is ...   
2  Easy to start up and does not overheat as much...   
3     Great laptop that offers many great features !   
4  One night I turned the freaking thing off afte...   

                                         annotations  
0                            [([16, 17], [15], POS)]  
1  [([4], [3], POS), ([9], [8], POS), ([26], [25]...  
2                               [([2, 3], [0], POS)]  
3                                  [([6], [5], POS)]  
4  [([21], [20], NEG), ([23], [25], NEG), ([27, 2...  


In [5]:
pd.set_option('display.max_colwidth', None)
print(df.head())

                                                                                                                                                                                               sentence  \
0                                                                                                              I charge it at night and skip taking the cord with me because of the good battery life .   
1                it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .   
2                                                                                                                                     Easy to start up and does not overheat as much as other laptops .   
3                                                                                                                                                        Great laptop that offers many great

In [6]:
print(df.loc[0, "sentence"])
print(df.loc[0, "annotations"])

I charge it at night and skip taking the cord with me because of the good battery life .
[([16, 17], [15], 'POS')]


In [7]:
rows = []

for _, row in df.iterrows():
    sentence = row["sentence"]
    for aspect, opinion, sentiment in row["annotations"]:
        rows.append({
            "sentence": sentence,
            "aspect": aspect,
            "opinion": opinion,
            "sentiment": sentiment
        })

df_flat = pd.DataFrame(rows)

print(df_flat.head(10))

                                                                                                                                                                                               sentence  \
0                                                                                                              I charge it at night and skip taking the cord with me because of the good battery life .   
1                it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .   
2                it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .   
3                it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely

In [8]:
def get_words(sentence, spans):
    tokens = sentence.split()
    return " ".join(tokens[i] for i in spans)

rows = []

for _, row in df.iterrows():
    sentence = row["sentence"]
    tokens = sentence.split()

    for aspect, opinion, sentiment in row["annotations"]:
        rows.append({
            "sentence": sentence,
            "aspect_text": get_words(sentence, aspect),
            "opinion_text": get_words(sentence, opinion),
            "sentiment": sentiment
        })

df_readable = pd.DataFrame(rows)
print(df_readable.head())

                                                                                                                                                                                 sentence  \
0                                                                                                I charge it at night and skip taking the cord with me because of the good battery life .   
1  it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .   
2  it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .   
3  it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .   
4  it is of high quality , has a killer GUI , is extrem

In [9]:
df_readable.shape

(1460, 4)

In [10]:
df_readable.head(100)

,sentence,aspect_text,opinion_text,sentiment
0,I charge it at night and skip taking the cord with me because of the good battery life .,battery life,good,POS
1,"it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .",quality,high,POS
2,"it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .",GUI,killer,POS
3,"it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .",applications,good,POS
4,"it is of high quality , has a killer GUI , is extremely stable , is highly expandable , is bundled with lots of very good applications , is easy to use , and is absolutely gorgeous .",use,easy,POS
...,...,...,...,...
95,"I wish it had a webcam though , then it would be perfect !",webcam,wish,NEG
96,Another thing I might add is the battery life is excellent .,battery life,excellent,POS
97,"One drawback , I wish the keys were backlit .",keys,drawback,NEG
98,I wish the volume could be louder and the mouse didnt break after only a month .,volume,louder,NEG


In [11]:
df["target"] = df_readable.apply(
    lambda x: f"({x.aspect_text}, {x.opinion_text}, {x.sentiment})",
    axis=1
)

grouped = df.groupby("sentence")["target"].apply(lambda x: "; ".join(x)).reset_index()
grouped.head()

,sentence,target
0,( No problem with the ordering or shipping by the way .,"(instructions, no, NEG)"
1,) And printing from either word processor is an adventure .,"(2 GB of RAM, plenty, POS)"
2,"-Called headquarters again , they report that TFT panel is broken , should be fixed by the end of the week ( week 3 ) .","(weight, bad, NEG)"
3,-Computer crashed frequently and battery life decreased very quickly .,"(customer services, bad, NEG)"
4,-I propose that they can just swap the hard drives .,"(space bar, noisy, NEG)"


In [12]:
from datasets import Dataset

dataset = Dataset.from_pandas(grouped)

dataset = dataset.train_test_split(test_size=0.1)

In [13]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [14]:
def preprocess(example):
    inputs = "ASTE: " + example["sentence"]
    targets = example["target"]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )["input_ids"]

    # IMPORTANT: replace pad tokens with -100
    labels = [
        -100 if token == tokenizer.pad_token_id else token
        for token in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs

In [15]:
# tokenized = dataset.map(
#     preprocess,
#     remove_columns=["sentence", "target"]
# )

In [16]:
tokenized = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [17]:
print(tokenized)
print(tokenized["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 810
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 90
    })
})
{'input_ids': [71, 15258, 10, 37, 1056, 19, 3, 2046, 29, 11904, 16, 82, 842, 3, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=30,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [30]:
from transformers import Trainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator
)

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.222519,2.118273
2,2.049961,2.073750
3,1.989724,2.049395
4,2.069565,2.021222
5,2.203402,1.998983
6,1.985062,1.979164
7,2.007958,1.961910
8,2.160957,1.949483
9,1.832599,1.937792
10,1.900604,1.930250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3060, training_loss=1.9304184701707627, metrics={'train_runtime': 149.5571, 'train_samples_per_second': 162.48, 'train_steps_per_second': 20.46, 'total_flos': 822201443942400.0, 'train_loss': 1.9304184701707627, 'epoch': 30.0})

In [32]:
trainer.save_model("./aste_model_final")
tokenizer.save_pretrained("./aste_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./aste_model_final\\tokenizer_config.json',
 './aste_model_final\\tokenizer.json')

# Inference 

In [33]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# =========================
# 1. Load model
# =========================

model_path = "./aste_model_final"   # OR checkpoint folder

tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# =========================
# 2. Predict
# =========================

def predict_aste(sentence):
    input_text = "ASTE: " + sentence

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# =========================
# 3. Parse
# =========================

def parse_aste_output(text):
    triples = []

    for item in text.split(";"):
        item = item.strip().strip("()")
        if not item:
            continue

        parts = [p.strip() for p in item.split(",")]
        if len(parts) == 3:
            triples.append({
                "aspect": parts[0],
                "opinion": parts[1],
                "sentiment": parts[2]
            })

    return triples


# =========================
# 4. Run full pipeline
# =========================

def run_aste(sentence):
    raw = predict_aste(sentence)
    return {
        "raw_output": raw,
        "triples": parse_aste_output(raw)
    }


# =========================
# 5. Test
# =========================

sentence = "The battery life is excellent but the screen is bad."

result = run_aste(sentence)

print("RAW OUTPUT:")
print(result["raw_output"])

print("\nTRIPLES:")
for t in result["triples"]:
    print(t)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

RAW OUTPUT:
(features, great, POS)

TRIPLES:
{'aspect': 'features', 'opinion': 'great', 'sentiment': 'POS'}
